<a href="https://colab.research.google.com/github/emrecnakts/U-FFIA/blob/main/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [42]:
!cd /content/drive/MyDrive/uufia/ufia/uff && zip -r /content/drive/MyDrive/uufia_backup_v2_$(date +%Y%m%d).zip . -x "*.pt" -x "data/*"

  adding: fusion_test.py (deflated 62%)
  adding: main.py (deflated 67%)
  adding: main_kl_unified.py (deflated 68%)
  adding: README.md (deflated 54%)
  adding: Unified.py (deflated 77%)
  adding: environment.yml (deflated 58%)
  adding: fish_feeding-min.png (deflated 1%)
  adding: main_video.py (deflated 66%)
  adding: .gitignore (deflated 55%)
  adding: pipline.png (deflated 9%)
  adding: spectrogram.png (deflated 0%)
  adding: main_unified.py (deflated 67%)
  adding: dataset/ (stored 0%)
  adding: dataset/fish_video_dataset.py (deflated 73%)
  adding: dataset/audio_dataset.py (deflated 71%)
  adding: dataset/unified_dataset.py (deflated 73%)
  adding: dataset/fish_audio_dataset.py (deflated 72%)
  adding: dataset/__pycache__/ (stored 0%)
  adding: dataset/__pycache__/fish_av_dataset.cpython-312.pyc (deflated 47%)
  adding: dataset/__pycache__/fish_av_dataset.cpython-313.pyc (deflated 47%)
  adding: dataset/fish_av_dataset.py (deflated 68%)
  adding: config/ (stored 0%)
  adding: co

In [41]:
from utils.pytorch_utils import forward_av
from scipy.special import softmax
import numpy as np
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix

# --- ADIM A: threshold'u VAL set uzerinden bul ---
val_output_dict = forward_av(model=model, generator=val_loader, return_target=True)
val_clipwise = np.asarray(val_output_dict['clipwise_output'])
val_target = np.asarray(val_output_dict['target'])
val_target_label = val_target.reshape(-1) if val_target.ndim == 1 else np.argmax(val_target, axis=1)
val_probs = softmax(val_clipwise, axis=1)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(val_target_label, val_probs)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1s)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print(f"Val uzerinden secilen threshold: {best_threshold:.4f} (val F1: {f1s[best_idx]:.4f})")

# --- ADIM B: bu sabit threshold'u TEST set'e tek seferlik uygula ---
test_output_dict = forward_av(model=model, generator=test_loader, return_target=True)
test_clipwise = np.asarray(test_output_dict['clipwise_output'])
test_target = np.asarray(test_output_dict['target'])
test_target_label = test_target.reshape(-1) if test_target.ndim == 1 else np.argmax(test_target, axis=1)
test_probs = softmax(test_clipwise, axis=1)[:, 1]

test_preds = (test_probs > best_threshold).astype(int)
print("\n=== TEST SONUCU (val'den secilen threshold ile) ===")
print(classification_report(test_target_label, test_preds))
print(confusion_matrix(test_target_label, test_preds))

Evaluation starting ...: 100%|██████████| 9/9 [00:39<00:00,  4.42s/it]


Val uzerinden secilen threshold: 0.7016 (val F1: 1.0000)


Evaluation starting ...: 100%|██████████| 17/17 [01:11<00:00,  4.22s/it]


=== TEST SONUCU (val'den secilen threshold ile) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       201
           1       1.00      0.99      1.00       132

    accuracy                           1.00       333
   macro avg       1.00      1.00      1.00       333
weighted avg       1.00      1.00      1.00       333

[[201   0]
 [  1 131]]


In [31]:
from utils.pytorch_utils import forward_av
from scipy.special import softmax
import numpy as np
from sklearn.metrics import precision_recall_curve, classification_report, confusion_matrix

output_dict = forward_av(model=model, generator=test_loader, return_target=True)
clipwise_output = np.asarray(output_dict['clipwise_output'])
target = np.asarray(output_dict['target'])
target_label = target.reshape(-1) if target.ndim == 1 else np.argmax(target, axis=1)
probs = softmax(clipwise_output, axis=1)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(target_label, probs)
f1s = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1s)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
print(f"En iyi threshold: {best_threshold:.3f}, F1: {f1s[best_idx]:.3f}")

preds_new = (probs > best_threshold).astype(int)
print(classification_report(target_label, preds_new))
print(confusion_matrix(target_label, preds_new))

Evaluation starting ...: 100%|██████████| 17/17 [01:19<00:00,  4.66s/it]

En iyi threshold: 0.003, F1: 0.992
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       201
           1       0.98      0.99      0.99       132

    accuracy                           0.99       333
   macro avg       0.99      0.99      0.99       333
weighted avg       0.99      0.99      0.99       333

[[199   2]
 [  1 131]]


In [28]:
model_path = "/content/drive/MyDrive/uufia/ufia/uff/Fish_workspace/audio-video-fusion(MBT-4L)/save_models/atten8_best.pt"
model.load_state_dict(torch.load(model_path, weights_only=False)['model_state_dict'])
model.eval()
from utils.evaluate import Evaluator
evaluator = Evaluator(model=model)
test_statistics = evaluator.evaluate_av(test_loader)
print("Test accuracy:", test_statistics['accuracy'])
print("Test mAP:", test_statistics['average_precision'])

Evaluation starting ...: 100%|██████████| 17/17 [01:45<00:00,  6.19s/it]

Test accuracy: 0.8618618618618619
Test mAP: 0.998579678412808


In [12]:
%cd /content/drive/MyDrive/uufia/ufia/uff
!python main_av.py --config config/audiovisual/exp2_av.yaml

/content/drive/MyDrive/uufia/ufia/uff
2026-08-22 11:13:19.572617: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 2026-08-22 11:13:27,914 - INFO - {'Exp_name': 'audio-video-fusion(MBT-4L)', 'Modality': 'audio_video', 'Data_dir': '../data', 'Training': {'Batch_size': 20, 'Max_epoch': 20, 'learning_rate': 0.001, 'seed': 25, 'classes_num': 2}, 'Audio_features': {'sample_rate': 64000, 'window_size': 2048, 'hop_size': 1024, 'mel_bins': 64, 'fmin': 1, 'fmax': 128000}, 'Model': {'video_name': 'S3D', 'audio_name': 'MobileNetV2'}, 'Workspace': 'Fish_workspace'}
 2026-08-22 11:13:27,915 - INFO - Audio_video_Model(
  (video_encoder): S3D(
    (base): Sequential(
      (0): SepConv3d(
        (conv_s): Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), 

In [ ]:
# 1. Eksik kütüphaneleri tekrar kur
!pip install torchlibrosa tensorboard matplotlib-inline

# 2. Doğru klasöre git
%cd /content/drive/MyDrive/uufia/ufia/uff/

# 3. Eğitimi başlat
!MPLBACKEND=Agg python main_av.py --config config/audiovisual/exp2_av.yaml

/content/drive/MyDrive/uufia/ufia/uff
2026-08-21 11:20:01.168998: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 2026-08-21 11:20:56,685 - INFO - {'Exp_name': 'audio-video-fusion(MBT-4L)', 'Modality': 'audio_video', 'Data_dir': '../data', 'Training': {'Batch_size': 20, 'Max_epoch': 20, 'learning_rate': 0.001, 'seed': 25, 'classes_num': 2}, 'Audio_features': {'sample_rate': 64000, 'window_size': 2048, 'hop_size': 1024, 'mel_bins': 64, 'fmin': 1, 'fmax': 128000}, 'Model': {'video_name': 'S3D', 'audio_name': 'MobileNetV2'}, 'Workspace': 'Fish_workspace'}
 2026-08-21 11:20:56,689 - INFO - Audio_video_Model(
  (video_encoder): S3D(
    (base): Sequential(
      (0): SepConv3d(
        (conv_s): Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), 

In [ ]:
!python main_av.py --config config/audiovisual/exp2_av.yaml

python3: can't open file '/content/main_av.py': [Errno 2] No such file or directory


In [25]:
%cd /content/drive/MyDrive/uufia/ufia/uff

/content/drive/MyDrive/uufia/ufia/uff


In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
